In [ ]:
import requests
import pandas as pd

# Replace with your NOAA API token
API_TOKEN = 'place token here' # Request token: https://www.ncdc.noaa.gov/cdo-web/token
BASE_URL = 'https://www.ncei.noaa.gov/cdo-web/api/v2/stations'

HEADERS = {'token': API_TOKEN}
STATES = ['CA', 'NY', 'KS']  # Replace with proper FIPS codes
DATASET_ID = 'GHCND'  # Daily summaries dataset

def fetch_stations(state):
    stations = []
    offset = 1
    limit = 1000

    while True:
        params = {
            'datasetid': DATASET_ID,
            'locationid': f'FIPS:{state}',
            'limit': limit,
            'offset': offset
        }
        response = requests.get(BASE_URL, headers=HEADERS, params=params)
        data = response.json()

        if 'results' not in data:
            break

        stations.extend(data['results'])
        if len(data['results']) < limit:
            break
        offset += limit

    return stations

# Collect and format data
all_stations = []
for state in STATES:
    print(f"Fetching stations for {state}...")
    stations = fetch_stations(state)
    for s in stations:
        all_stations.append({
            'ID': s.get('id'),
            'Name': s.get('name'),
            'State': state,
            'Latitude': s.get('latitude'),
            'Longitude': s.get('longitude')
        })

df = pd.DataFrame(all_stations)
print(df.head())